In [1038]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score,balanced_accuracy_score, recall_score

### Load the embeddings for the C.S.sylv sulcal region

In [282]:
#ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv', index_col=0)
ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv', index_col=0)
print(ukb_embeddings.shape)
ukb_embeddings.head()

(42433, 256)


,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim247,dim248,dim249,dim250,dim251,dim252,dim253,dim254,dim255,dim256
ID,,,,,,,,,,,,,,,,,,,,,
sub-1000021,-37.629620,6.583590,29.279840,-16.611984,-30.081190,-24.476210,11.042302,-6.784241,19.132568,10.150197,...,-2.848078,11.085325,-12.354747,-76.037210,-8.468971,-29.602938,-24.906027,-16.344492,7.703534,-1.050617
sub-1000325,11.934219,11.355382,-13.496794,-38.996407,14.297873,-5.811631,6.174108,-45.331050,9.762536,44.107750,...,4.212949,6.075599,-36.018467,-134.833560,26.447592,-35.490543,4.641970,-4.289551,-58.478560,-31.515226
sub-1000458,-25.394463,4.755465,-22.513622,-25.682335,-67.619896,-18.027046,-30.610876,51.657852,-18.840885,-7.459018,...,-9.459454,-0.912658,32.695583,12.509913,10.790437,-25.991050,-39.407890,-13.558734,47.675068,-0.354762
sub-1000575,-25.367662,-23.005732,-30.035063,36.237038,-103.579190,-9.643719,4.125421,34.603440,19.479185,-13.243917,...,20.220034,-5.049628,4.961305,-25.468930,25.991018,3.052118,-73.673270,6.855866,30.889896,43.402153
sub-1000606,-35.599197,-2.058082,26.222765,-3.495081,-49.495464,-3.061627,-14.236676,-3.899254,-15.054834,50.399190,...,8.652889,27.369568,34.453293,-73.026400,40.409150,28.420736,-51.241203,17.027500,-0.707319,-20.734170


### Reduce dimension (hope to remove the noise) with a PCA

In [1078]:
n_components=19

pca = PCA(n_components=n_components)
pca.fit(ukb_embeddings)
print(pca.explained_variance_ratio_)
(np.cumsum(pca.explained_variance_ratio_) < 0.99).sum()

[0.16603878 0.1217284  0.11188194 0.11059644 0.09332536 0.08971151
 0.08181378 0.05803272 0.04799348 0.03264414 0.02280461 0.01543608
 0.01228882 0.00817237 0.00651911 0.0047046  0.00291734 0.00251397
 0.00186247]


18

In [1079]:
ukb_pca_bdd = pca.transform(ukb_embeddings)

In [1080]:
#scaler = StandardScaler()
#scaler.fit(ukb_embeddings)
#ukb_scl_bdd = scaler.transform(ukb_embeddings)
#ukb_scl_bdd

#### First approach: SVM trained to find the interruption

In [1081]:
model = SVC(kernel='linear', probability=True,
            random_state=42,
            C=0.01, class_weight='balanced', decision_function_shape='ovr')

In [1082]:
interrupted = [
'sub-1310920',
'sub-1376904',
'sub-2863742',
'sub-3694216',
'sub-1037052',
'sub-3250551',
'sub-5401486',
'sub-1499791', # good
'sub-1911266',
'sub-4217758',
'sub-2693192',
'sub-1633860',
'sub-5222070',
'sub-3292254',
'sub-1613821',
'sub-2771619',
'sub-3159828',
'sub-4632483',
'sub-5936108',
'sub-3794487',
'sub-1420697', # not sure
'sub-1111996', # not sure
'sub-1425827', # not sure
'sub-2846621', # good
'sub-2004479',
'sub-3891499',
'sub-5236788',
'sub-3061407', # very good
'sub-5693167',
'sub-2155264', # very good
'sub-2444973', # very good
'sub-5245412', # good
'sub-5574911', # very good
'sub-2852894', # very good
'sub-1106033', # very good
'sub-5984646', # very good
'sub-5739487', # very good
'sub-3492298', # good
'sub-5712569', # not sure
'sub-2200121', # not sure
'sub-5638090', # good
'sub-4496792', # good
'sub-5129881', # good
'sub-1775041', # good
'sub-1094593', # good
'sub-1358401', # good
'sub-4354208', # very good
'sub-1428452', # good
'sub-5731125',
'sub-4995189', # very good
'sub-1499791',
'sub-2762943', # very good
'sub-3386408', # not sure
'sub-5665554', # not sure
'sub-1130686', # good
'sub-2484762', # good
'sub-5186095', # good
'sub-5569356',
'sub-4762603', # not sure
'sub-3572724', # good
'sub-2573795', # good
'sub-5315648',
'sub-2731992',
'sub-4949978',
'sub-2776534',
'sub-2298245',
'sub-2570335',
'sub-3258249', # not sure
'sub-1748817', # not sure
'sub-4203366', # very good
'sub-4184635', # very good
'sub-1053493', # note sure
'sub-3947538', # not sure
'sub-2741631', # not sure
'sub-3492301',
'sub-4447456',
'sub-2373286', # not sure
'sub-4732282',
'sub-3293670',
'sub-2149638', # good
'sub-4625643', # not sure
'sub-4328267',
'sub-2589361',
'sub-4232003',
'sub-5456948',
'sub-4420000', # not sure
'sub-1553423', # not sure
'sub-1405899', # not sure
'sub-2550690', # not sure
'sub-2986522', # not sure
'sub-1698233', # not sure
'sub-4603077', # not sure
'sub-3428215',
'sub-1935008',
'sub-4589882',
'sub-2323818',
'sub-2230154',
'sub-1675253',
'sub-4875056',
'sub-4059279',
'sub-4067363',
'sub-1322441',
'sub-1417407',
'sub-3733675',
'sub-5531350',
'sub-4067363',
'sub-1369171',
'sub-1807186',
'sub-1267836',
'sub-3758439',
'sub-4652131',
'sub-1052521',
'sub-5949398',
'sub-3672666',
'sub-4754998',
'sub-3791185',
'sub-4587270',
'sub-4599903',
'sub-5617588'
]

not_interrupted = [
'sub-3264612', 
'sub-4805237', 
'sub-1422413',
'sub-3264612', 
'sub-4805237', 
'sub-1422413',
'sub-4491384', 
'sub-2946274',
'sub-5581707',
'sub-4834994',
'sub-5437419',
'sub-5054716',
'sub-2889389',
'sub-4520944',
'sub-3009279',
'sub-1190643',
'sub-5123219',
'sub-4016129',
'sub-4411765',
'sub-3234836',
'sub-5486726',
'sub-2592717',
'sub-4116944',
'sub-3670173',
'sub-1273718',
'sub-2833426',
'sub-1352284',
'sub-2389411', 
'sub-2970418', 
'sub-5605784', 
'sub-2141551', 
'sub-1979982',
'sub-5643778', 
'sub-3693543', 
'sub-4805119', 
'sub-5686761', 
'sub-2733674',
'sub-2097565',
'sub-5292898', 
'sub-2118136',
'sub-5966409',
'sub-1996092',
'sub-2036033',
'sub-3333294',
'sub-2193253',
'sub-3603191',
'sub-3936967', 
'sub-1286007', 
'sub-3013938',
'sub-5117110',
'sub-2228486',
'sub-3721299',
'sub-4420611', 
'sub-2349203',
'sub-2207793', 
'sub-2816262',
'sub-3765466',
'sub-2957401',
'sub-5998652',
'sub-2837393',
'sub-5836983',
'sub-1167379',
'sub-5729132', 
'sub-3453064',
'sub-3334219',
'sub-5082433',
'sub-2337820',
'sub-3994474',
'sub-2968297',
'sub-1701563',
'sub-4536778',
'sub-5723111',
'sub-3227039',
'sub-5147403',
'sub-3627711',
'sub-1465129',
'sub-2802489',
'sub-3008660',
'sub-5027399',
'sub-1103646',
'sub-4741296',
'sub-5749108',
'sub-1398736',
'sub-2741815', 
'sub-3722413', 
'sub-5217534', 
'sub-2583027',
'sub-3388306',
'sub-4755899',
'sub-5754849',
'sub-3388080',
'sub-5578922',
'sub-5237880',
'sub-3388306', 
'sub-5237880', 
'sub-3379262', 
'sub-3529189',
'sub-3541105',
'sub-2834970',
'sub-3992259',
'sub-4519441',
'sub-2538754',
'sub-2420937',
'sub-4027732', 
'sub-2005939',
'sub-4428393',
'sub-2284024',
'sub-1298876',
'sub-4791977',
'sub-3401499', 
'sub-5430535', 
'sub-4787289',
'sub-1734788',
'sub-4727825', 
'sub-1597706', 
'sub-4868991', 
'sub-5649675', 
'sub-2427515',
'sub-5293703', 
'sub-5319071', 
'sub-3525594', 
'sub-5561142'
] 

In [1083]:
X = ukb_embeddings.loc[interrupted + not_interrupted]
y = [1 for i in range(len(interrupted))] + [0 for i in range(len(not_interrupted))]
X_pca = pca.transform(X)
len(interrupted), len(not_interrupted)

(119, 122)

In [1088]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.33, random_state=42)
model.fit(X_train_pca, y_train)

print('Recall:', recall_score(y_test, model.predict(X_test_pca)), '\n')
print('ROC:', roc_auc_score(y_test ,model.predict_proba(X_test_pca)[:,1]), '\n')
print('Balanced accuracy:',balanced_accuracy_score(y_test, model.predict(X_test_pca)), '\n')
model.fit(X_pca, y)

Recall: 0.825 

ROC: 0.8675 

Balanced accuracy: 0.8125 



SVC(C=0.01, class_weight='balanced', kernel='linear', probability=True,
    random_state=42)

In [1089]:
prediction = pd.DataFrame({"IID" : list(ukb_embeddings.index),
              "Pred" : model.predict_proba(ukb_pca_bdd)[:,1]})
prediction

,IID,Pred
0,sub-1000021,0.051490
1,sub-1000325,0.607315
2,sub-1000458,0.322983
3,sub-1000575,0.125217
4,sub-1000606,0.183764
...,...,...
42428,sub-6023847,0.062501
42429,sub-6024038,0.117524
42430,sub-6024150,0.022434
42431,sub-6024379,0.152262


In [1094]:
print('Maximum probability of prediction among the interrupted C.S. :',prediction[prediction["IID"].isin(interrupted)].Pred.max(), '\n')
print('Mean probability of prediction among the interrupted C.S. :',prediction[prediction["IID"].isin(interrupted)].Pred.mean(), '\n')
prediction[prediction['IID']=='sub-2036033']

Maximum probability of prediction among the interrupted C.S. : 0.933579246792166 

Mean probability of prediction among the interrupted C.S. : 0.6810307167062286 



,IID,Pred
8695,sub-2036033,0.263998


In [1095]:
((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred")[-5:].IID).to_list()

['sub-2204575', 'sub-2969851', 'sub-3652752', 'sub-1266896', 'sub-5801489']

#### Second approach: Euclidian distance in the reduced latent space

In [1096]:
from scipy.spatial import distance

In [1097]:
list_dist = [distance.euclidean(pca.transform(ukb_embeddings.loc['sub-3791185'].to_numpy().reshape(1,-1)), ukb_pca_bdd[i]) for i in range(len(ukb_pca_bdd))]
df_dist = pd.DataFrame({"IID":list(ukb_embeddings.index), "Dist":list_dist})

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr

In [1098]:
sample_dist = ((df_dist[~(df_dist["IID"].isin(interrupted))]).sort_values(by='Dist').iloc[22000:22025].IID).to_list()

### Visualization with Anatomist

In [1099]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims

In [1100]:
dataset = 'UkBioBank40'
region = "S.C.-sylv."
side = "R"

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'

In [1101]:
sample = ((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred", ascending=False)[25:50].IID).to_list()

In [1102]:
volume=True
volume_files = []

for subject_id in sample:
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
    
    if volume:
        if os.path.isfile(volume_path):
            vol = aims.read(volume_path)
            volume_files.append(vol)
        else:
            print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

block = a.createWindowsBlock(5) # 10 columns
dic_windows = {}

if volume:
    for i, vol in enumerate(volume_files):
        dic_windows[f'a_vol{i}'] = a.toAObject(vol)
        #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
        dic_windows[f'rvol{i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{i}']], method='VolumeRenderingFusionMethod')
        dic_windows[f'rvol{i}'].releaseAppRef()
        dic_windows[f'wvr{i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'wvr{i}'].addObjects(dic_windows[f'rvol{i}'])

In [1103]:
sample[14]

'sub-2285914'

In [1077]:
sample_dist[15:19]

['sub-5293703', 'sub-5319071', 'sub-3525594', 'sub-5561142']